# 프롬프트(prompt)

프롬프트는 사용자와 언어 모델 간의 대화에서 질문이나 요청의 형태로 제시되는 입력문이다. 모델이 어떤 유형의 응답을 제공할지 결정하는데 중요한 역할을 한다.

In [1]:
import os

# os.environ['OPENAI_API_KEY'] = '사용자 API Key'


## PromptTemplate

프롬프트 템플릿은 단일 문장 또는 간단한 명령을 입력하여 단일 문장 또는 간단한 응답을 생성하는 데 사용되는 프롬프트를 구성할 수 있는 문자열 템플릿이다. 파이썬의 문자열 포맷팅을 사용하여 동적으로 특정한 위치에 입력값을 포함시킬 수 있다.

구성요서  
LLM 모델에 입력할 프롬프트를 구성할 때 `지시`, `예시`, `맥락`, `질문`과 같은 다양한 구성 요소들을 조합할 수 있다. 다양한 시나리오에서 필요한 구성 요소들을 조합해서 적용한다.

`지시`: 언어 모델에게 어떤 작업을 수행하도록 요청하는 구체적인 지시.  
&nbsp;&nbsp;&nbsp;▶ 아래 제공된 제품 리뷰를 요약해 줘.  
`예시`: 요청된 작업을 수행하는 방법에 대한 하나 이상의 예시.  
&nbsp;&nbsp;&nbsp;▶ 예를 들어, '이 제품은 매우 사용하기 편리하며 배터리 수명이 길다.'라는 리뷰는 '사용 편의성과 긴 배터리 수명이 특징'으로 요약한다.  
`맥락`: 특정 작업을 수행하기 위한 추가적인 맥락.  
&nbsp;&nbsp;&nbsp;▶ 리뷰는 스마트 위치에 대한 것이며, 사용자 경험에 초점을 맞추고 있다.  
`질문`: 어떤 답변을 요구하는 구체적인 질문.  
&nbsp;&nbsp;&nbsp;▶ 이 리뷰를 바탕으로 스마트 워치와 주요 장점을 두 세 문장으로 요약해 줘.

================================================================================================================================================

LangChain에서 PromptTemplate과 ChatPromptTemplate은 모두 프롬프트를 동적으로 생성하기 위한 도구이지만, 대상으로 하는 LLM의 유형과 메시지 구조에서 차이있다.

`구분            PromptTemplate                              ChatPromptTemplate                          `  
`주요 대상       텍스트 완성형 모델(Completion LLMs)         대화형 모델(Chat Models)                    `  
`                 (예: GPT-3, legacy text-davinci)            (예: GPT-4o, Claude 3.5, Gemini 1.5)         `  
`출력 형태       단일 문자열(String)                         메시지 객체의 리스트(List of BaseMessage)   `  
`입력 구조       단일 프롬프트 텍스트                        역할(Role)별 메시지 조합(System, Human, AI) `  
`주요 사용처     단답형 요약, 단순 텍스트 생성               다단계 대화, 역할 부여가 필요한 에이전트/RAG`

AI 모델(LLM)에 전달할 프롬프트(질문이나 지시사항)를 만들기 위해 import 한다.

PromptTemplate은 변수를 넣을 수 있는 프롬프트 템플릿(양식)이다. 매번 AI에게 긴 질문을 새로 작성할 필요 없이, 고정된 틀(template)을 만들어 두고 바뀌는 부분만 변수로 끼워 넣을 수 있게 해준다.

재사용성: 동일한 구조의 프롬프트를 여러 데이터에 반복해서 적용할 수 있다.  
동적 생성: 사용자 입력이나 데이터베이스 값에 따라 프롬프트 내용을 조립한다.  
코드의 깔끔함: 문자열을 '+'로 연결하거나 f스트링으로 복잡하게 처리하던 프롬프트 조작을 표준화된 방식으로 관리한다.

In [2]:
# 고정된 텍스트 내부에 변수를 지정해서 원하는 값을 동적으로 채워넣을 수 있다.
from langchain_core.prompts import PromptTemplate

단일 문장 입력 => 단일 문장 출력

문자열 프롬프트를 위한 템플릿을 생성한다. 파이썬의 문자열 포맷팅 구문을 사용한다.

In [3]:
# 템플릿 문장 정의
# 실제 프롬프트로 사용할 문자열을 정의한다.
# 여기서 중괄호 {name}과 {age}는 나중에 데이터가 들어갈 변수 자리이다. 파이썬의 f스트링 형식과 유사하다.
template_text = '안녕하세요, 제 이름은 {name}이고, 나이는 {age}살 입니다.'

# 프롬프트 템플릿 객체 생성
# from_template() 메소드를 사용해서 template_text에 저정된 문자열을 LLM이 이해할 수 있는 템플릿 객체를 생성한다.
# 이 객체는 어떤 변수(name, age)가 필요한지 자동으로 분석하고 관리한다.
prompt_template = PromptTemplate.from_template(template_text)

# 프롬프트에 데이터 채워넣기(formatting)
# format() 메소드를 사용해서 템플릿의 비워둔 자리({name}과 {age})에 실제값을 전달한다.
# name에는 '홍길동'을, age에는 30을 대입해서 최종 프롬프트를 완성한다.
filled_prompt = prompt_template.format(name='홍길동', age=30)
filled_prompt

'안녕하세요, 제 이름은 홍길동이고, 나이는 30살 입니다.'

템플릿 결합

여러 개의 프롬프트 조각이나 문자열을 '+' 연산자로 합친다.

In [4]:
combined_prompt = (
    # 기존 템플릿 가져오기
    # 앞서 만든 첫 번째 템플릿('안녕하세요, 제 이름은 {name}...')을 첫 조각으로 사용한다.
    # 현재까지 필요한 변수는 {name}과 {age}이다.
    prompt_template
    # 새로운 템플릿 연결하기
    # 새로운 PromptTemplate 객체를 만들어서 기존 템플릿인 prompt_template 뒤에 연결한다.
    # 기존 템플릿 끝에 줄바꿈 2번과 함께 '아버지를 아버지라 부를 수 없습니다.'라는 문자열을 추가한다.
    + PromptTemplate.from_template('\n\n아버지를 아버지라 부를 수 없습니다.')
    # 일반 문자열 연결하기
    # LLM은 PromptTemplate 객체와 일반 문자열을 '+' 연산자로 연결하면, 자동으로 전체를 하나의 커다란 PromptTemplate 객체로 만든다.
    # {language}라는 새로운 변수가 추가된다.
    + '\n\n{language}로 번역해 주세요.'
)

# 이 객체는 이제 총 3개의 변수({name}, {age}, {language})를 입력받아야 하는 하나의 통합 템플릿이 된다.
combined_prompt

PromptTemplate(input_variables=['age', 'language', 'name'], input_types={}, partial_variables={}, template='안녕하세요, 제 이름은 {name}이고, 나이는 {age}살 입니다.\n\n아버지를 아버지라 부를 수 없습니다.\n\n{language}로 번역해 주세요.')

format() 메소드의 역할

PromptTemplate 객체로 생성한 템플릿 문자열 속에 중괄호 {}로 표시해두었던 변수 자리를 찾아 실제값으로 치환한다. 미리 짜놓은 '설문지 양식'에 실제 '사용자 정보'를 기입해서 제출용 문서를 만드는 과정으로 볼 수 있다.

파라미터 전달  
기존 템플릿과 새로운 템플릿 및 일반 문자열이 combined_prompt로 합쳐지면서 생성된 3개의 변수를 각각 매칭시킨다.  
name='홍길동': 템플릿 안의 {name} 자리에 '홍길동'이라는 문자열을 넣어준다.  
age=30: 템플릿 안의 {age} 자리에 숫자 30을 넣어준다. 내부적으로는 문자열 '30'으로 변환된다.  
language='영어': 일반 문자열에 있었던 {language} 변수에 '영어'라는 문자열을 넣어준다.

In [5]:
combined_prompt.format(name='홍길동', age=30, language='영어')

'안녕하세요, 제 이름은 홍길동이고, 나이는 30살 입니다.\n\n아버지를 아버지라 부를 수 없습니다.\n\n영어로 번역해 주세요.'

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [7]:
# 모델 설정
llm = ChatOpenAI(model='gpt-4o-mini')

# 체인 생성
chain = combined_prompt | llm | StrOutputParser()

# 체인 실행
print(chain.invoke({'name': '홍길동', 'age': 30, 'language': '영어'}))

Hello, my name is Hong Gil-dong, and I am 30 years old.

I cannot call my father "father."


## ChatPromptTemplate

ChatPromptTemplate은 대화형 상황에서 여러 메시지 입력을 기반으로 단일 메시지 응답을 생생하는데 사용되고 대화형 모델이나 챗봇 개발에 주로 사용한다. 입력은 여러 메시지를 원소로 갖는 리스트로 구성되며, 각 메시지는 역할(role)과 내용(content)으로 구성된다.

메시지 유형  
`SystemMessage`: 시스템의 기능을 설명한다.  
`HumanMessage`: 사용자 질문을 의미한다.  
`AIMessage`: AI 모델의 응답 결과이다.  
`FunctionMessage`: 특정 함수를 호출한 결과이다.  
`ToolMessage`: 특정 도구를 호출한 결과이다.

여러 메시지 입력 => 단일 메시지 출력

채팅 메시지를 원소로 갖는 리스트 형태로 각 채팅 메시지는 역할과 내용이 짝을 이루는 튜플 형태의 메시지 목록으로 프롬프트가 생성된다.

PromptTemplate과 달리 대화형 모델의 구조에 최적화된 ChatPromptTemplate는 챗봇은 만들 때 표준으로 사용된다.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

In [9]:
# 대화 구조 정의
# 대화의 흐름을 리스트 안에 튜플(역할, 내용) 형태로 정의한다.
# system: AI에게 부여하는 페르소나(정체성)이다.
# user: 실제 사용자가 AI에게 보낼 메시지이다.
chat_prompt = ChatPromptTemplate.from_messages([
    ('system', '이 시스템은 천문학 질문에 답변할 수 있습니다.'),
    ('user', '{user_input}'),
])
# print(chat_prompt)

# 메시지 생성
# format() 메소드는 단순히 하나의 문자열을 리턴하므로 PromptTemplate에서 사용하고 ChatPromptTemplate은 format_messages() 메소드를 사용한다.
# format_messages() 메소드는 메시지 객체들의 리스트를 리턴한다.
# message = chat_prompt.format(user_input='태양계에서 가장 큰 행성은 무엇인지 알려줘')
# message = chat_prompt.format_messages(user_input='태양계에서 가장 큰 행성은 무엇인지 알려줘')
# print(message)

# 체인 생성
chain = chat_prompt | llm | StrOutputParser()

# 체인 실행
chain.invoke({'user_input': '태양계에서 가장 큰 행성은 무엇인지 알려줘'})

'태양계에서 가장 큰 행성은 목성(Jupiter)입니다. 목성은 지구의 약 1,300배에 달하는 막대한 크기를 가지고 있으며, 수소와 헬륨으로 주로 구성된 거대한 가스 행성입니다.'

## MessagePromptTemplate

시스템 메시지용 템플릿과 사용자 메시지용 템플릿을 사용하기 위해 SystemMessagePromptTemplate과 HumanMessagePromptTemplate를 import 한다.  
SystemMessagePromptTemplate는 AI의 페르소나(역할, 정체성)을 설정한다. 배경 지식, 배경 지시사항  
HumanMessagePromptTemplate는 사용자가 AI에게 던지는 질문이나 실행할 요청을 설정한다.  
단순히 텍스트만 넣는게 아니라, 이 텍스트가 '시스템 지침'인지 '사용자의 질문'인지 타입을 확실하게 정의한다.

In [10]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [11]:
# 대화의 흐름을 리스트 안에 SystemMessagePromptTemplate, HumanMessagePromptTemplate 객체로 정의한다.
chat_prompt = ChatPromptTemplate.from_messages([
    # AI 에게 '너는 천문학 전문가야'라고 역할을 부여하는 시스템 메시지 템플릿을 만든다.
    SystemMessagePromptTemplate.from_template('이 시스템은 천문학 질문에 답변할 수 있습니다.'),
    # 사용자가 AI에게 물어볼 질문이 들어갈 {user_input} 변수가 포함된 사용자 메시지 템플릿을 만든다.
    HumanMessagePromptTemplate.from_template('{user_input}')
])
# print(chat_prompt)

# 메시지 생성
# message = chat_prompt.format_messages(user_input='태양계 위성에서 명왕성이 퇴출된 이유를 알려줘')
# print(message)

# 체인 생성
chain = chat_prompt | llm | StrOutputParser()

# 체인 실행
chain.invoke({'user_input': '태양계 위성에서 명왕성이 퇴출된 이유를 알려줘'})

"명왕성이 태양계의 행성 목록에서 퇴출된 이유는 주로 천문학자들이 정의한 '행성'의 기준을 충족하지 않기 때문입니다. 2006년 국제천문연맹(IAU)은 행성을 정의하는 세 가지 조건을 제정했습니다:\n\n1. 태양 주위를 공전해야 한다.\n2. 중력에 의해 구형 형태를 유지해야 한다.\n3. 자신의 궤도 주변을 청소해야 한다.\n\n명왕성은 첫 두 조건을 충족하지만, 세 번째 조건인 '자신의 궤도 주변을 청소'하는 것에는 실패합니다. 즉, 명왕성의 궤도 주변에는 다른 많은 물체들이 존재합니다. 따라서 명왕성은 '왜소행성'으로 분류되게 되었고, 이는 퇴출의 주된 이유입니다."